In [1]:
import torch
import torch.nn as nn
import torch.fx as fx
from chop import MaseGraph
import chop.passes as passes
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from chop.tools import get_logger, get_trainer, get_tokenized_dataset
from pathlib import Path

/vol/bitbucket/hv122/adls-project/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/vol/bitbucket/hv122/adls-project/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def get_trainer_with_split(model, tokenized_dataset, tokenizer, eval_split="validation_matched"):
    import numpy as np
    import evaluate
    from transformers import Trainer, TrainingArguments, DataCollatorWithPadding

    metric = evaluate.load("accuracy")

    def compute_accuracy(eval_pred):
        logits, labels = eval_pred

        if isinstance(logits, tuple):
            logits = logits[0]
            
        predictions = np.argmax(logits, axis=-1)
        return metric.compute(predictions=predictions, references=labels)

    training_args = TrainingArguments(
        "mase-trainer",
        report_to="none",
        num_train_epochs=1,
        save_strategy="no",
        warmup_steps=500,
        learning_rate=1e-5,
        disable_tqdm=True,
    )

    return Trainer(
        model,
        training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset[eval_split],
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        tokenizer=tokenizer,
        compute_metrics=compute_accuracy,
    )


def make_eval_fn(dataset, tokenizer, device, eval_split="validation_matched"):
    def eval_fn(model):
        trainer = get_trainer_with_split(
            model=model.to(device),
            tokenized_dataset=dataset,
            tokenizer=tokenizer,
            eval_split=eval_split,
        )
        metrics = trainer.evaluate()
        return metrics["eval_accuracy"]
    return eval_fn

Load dataset for bert-tiny!

In [3]:
tiny_model = AutoModelForSequenceClassification.from_pretrained("M-FAC/bert-tiny-finetuned-mnli")

from datasets import load_dataset
 
model_name = "M-FAC/bert-tiny-finetuned-mnli"
model = AutoModelForSequenceClassification.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
device = "cuda" if torch.cuda.is_available() else "cpu"
 
dataset = load_dataset("glue", "mnli")
 
def tokenize_fn(example):
    return tokenizer(example["premise"], example["hypothesis"], truncation=True, max_length=128)
 
tokenized_dataset = dataset.map(tokenize_fn, batched=True)

Evaluate the accuracy of the model on the MNLI dataset!

In [4]:
eval_fn = make_eval_fn(tokenized_dataset, tokenizer, device)
accuracy = eval_fn(model)
print(f"\nBert-tiny-mnli accuracy on MNLI validation_matched: {accuracy:.4f}")

/tmp/ipykernel_2215997/1645952674.py:27: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  return Trainer(


{'eval_loss': 0.7093204855918884, 'eval_model_preparation_time': 0.0004, 'eval_accuracy': 0.6990320937340805, 'eval_runtime': 2.6869, 'eval_samples_per_second': 3652.923, 'eval_steps_per_second': 456.662}

Bert-tiny-mnli accuracy on MNLI validation_matched: 0.6990


Execute the GELU to RELU swap!

In [5]:
from chop.tools import get_logger
 
logger = get_logger("mase_logger")
logger.setLevel("INFO")
 
def swap_gelu_for_relu(mg, pass_args={}):
    count = 0
    for node in mg.fx_graph.nodes:
        if node.op == "call_module":
            module = mg.modules[node.target]
            if isinstance(module, torch.nn.GELU):
                logger.info(f"Replacing GeLU with ReLU: {node.target}")
                parent_name, _, attr_name = node.target.rpartition(".")
                parent_module = mg.modules[parent_name] if parent_name else mg.model
                setattr(parent_module, attr_name, torch.nn.ReLU())
                count += 1
            else:
                logger.debug(f"Skipping module node: {node.target}")
        elif node.op == "call_function" and node.target in (torch.nn.functional.gelu,):
            logger.info(f"Replacing functional gelu call at: {node.name}")
            with mg.fx_graph.inserting_after(node):
                new_node = mg.fx_graph.call_function(torch.nn.functional.relu, args=(node.args[0],))
                node.replace_all_uses_with(new_node)
                mg.fx_graph.erase_node(node)
            count += 1
        else:
            logger.debug(f"Skipping node: {node.target}")
 
    mg.fx_graph.lint()
    logger.info(f"Replaced {count} GeLU activation(s) with ReLU")
    return mg, {}

MaseGraph handler

In [6]:
import sys
import os

def _to_masegraph(model):
    device = next(model.parameters()).device
    model.config.use_cache = False

    dummy_input = {
        "input_ids": torch.ones(1, 128, dtype=torch.long, device=device),
        "attention_mask": torch.ones(1, 128, dtype=torch.long, device=device),
        "labels": torch.zeros(1, dtype=torch.long, device=device),
    }

    mg = MaseGraph(model, hf_input_names=["input_ids", "attention_mask", "labels"])
    mg, _ = passes.init_metadata_analysis_pass(mg)

    with open(os.devnull, "w") as devnull:
        try:
            sys.stdout = devnull
            mg, _ = passes.add_common_metadata_analysis_pass(
                mg,
                pass_args={"dummy_in": dummy_input, "add_value": False},
            )
        finally:
            sys.stdout = sys.__stdout__

    return mg

In [9]:
save_path = "/vol/bitbucket/hv122/adls-data/bert-tiny-finetuned-mnli-model-no-gelu"


tokenizer = AutoTokenizer.from_pretrained(model_name)

model.cpu()
mg = _to_masegraph(model)
mg, _ = swap_gelu_for_relu(mg)

dummy_input = {
    "input_ids": torch.ones(1, 128, dtype=torch.long),
    "attention_mask": torch.ones(1, 128, dtype=torch.long),
    "labels": torch.zeros(1, dtype=torch.long),
}
mg, _ = passes.init_metadata_analysis_pass(mg)
mg, _ = passes.add_common_metadata_analysis_pass(
    mg, pass_args={"dummy_in": dummy_input, "add_value": False},
)
mg.export(save_path)

mg_loaded = MaseGraph.from_checkpoint(save_path)
for node in mg_loaded.fx_graph.nodes:
    if "gelu" in str(node.target).lower() or "relu" in str(node.target).lower():
        print(f"{node.op}: {node.target}")

for name, module in mg_loaded.model.named_modules():
    if "act" in name.lower() or "gelu" in name.lower() or "relu" in name.lower():
        print(f"{name}: {type(module)}")
accuracy = eval_fn(mg_loaded.model)
print(f"ReLU-swapped accuracy: {accuracy:.4f}")

INFO     Replacing functional gelu call at: gelu
INFO     Replacing functional gelu call at: gelu_1
INFO     Replaced 2 GeLU activation(s) with ReLU
INFO     Exporting MaseGraph to /vol/bitbucket/hv122/adls-data/bert-tiny-finetuned-mnli-model-no-gelu.pt, /vol/bitbucket/hv122/adls-data/bert-tiny-finetuned-mnli-model-no-gelu.mz
INFO     Exporting GraphModule to /vol/bitbucket/hv122/adls-data/bert-tiny-finetuned-mnli-model-no-gelu.pt
INFO     Saving full model format
INFO     Exporting MaseMetadata to /vol/bitbucket/hv122/adls-data/bert-tiny-finetuned-mnli-model-no-gelu.mz
/tmp/ipykernel_2215997/1645952674.py:27: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  return Trainer(


In [8]:
print("\n--- Architecture Verification ---")
gelu_count = 0
relu_count = 0

# Iterate through all modules in the loaded MaseGraph model
for name, module in mg_loaded.model.named_modules():
    if isinstance(module, torch.nn.GELU):
        gelu_count += 1
    elif isinstance(module, torch.nn.ReLU):
        relu_count += 1
        
print(f"GELU modules found: {gelu_count}")
print(f"ReLU modules found: {relu_count}")

if gelu_count > 0:
    raise ValueError(f"Verification Failed: Found {gelu_count} GELUs still in the model!")
else:
    print("Verification Passed: 0 GELUs detected. Proceeding to retrain.")
print("---------------------------------\n")

# trainer = get_trainer_with_split(
#     model=mg_loaded.model.to(device),
#     tokenized_dataset=tokenized_dataset,
#     tokenizer=tokenizer, 
#     eval_split="validation_matched",
# )

# trainer.train()

# metrics = trainer.evaluate()
# print(f"\nReLU-swapped retrained accuracy: {metrics['eval_accuracy']:.4f}")

In [23]:
metrics = trainer.evaluate()
print(f"\nReLU-swapped retrained accuracy: {metrics['eval_accuracy']:.4f}")

In [24]:
print("Raw metrics:", metrics)
print(f"Accuracy: {metrics.get('eval_accuracy', 'NOT FOUND')}")

In [25]:
from IPython.display import display

metrics = trainer.evaluate()
display(metrics)

{'eval_loss': 0.8622453808784485,
 'eval_accuracy': 0.6059093224656139,
 'eval_runtime': 2.1532,
 'eval_samples_per_second': 4558.264,
 'eval_steps_per_second': 569.841,
 'epoch': 1.0}